In [12]:
import torch
from torch import nn

In [13]:
import torch
import torch.nn as nn

class CustomLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, elementwise_affine=True):
        """
        Custom implementation of Layer Normalization.

        Args:
        - normalized_shape: the dimension(s) to normalize
        - eps: a small constant added for numerical stability (to prevent division by zero)
        - elementwise_affine: whether to use learnable scaling (γ) and shifting (β) parameters
        """
        super().__init__()
        
        # Check whether normalized_shape is an int or a tuple
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = tuple(normalized_shape)
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        
        if self.elementwise_affine:
            # Learnable scaling parameter γ, initialized to 1
            self.gamma = nn.Parameter(torch.ones(*self.normalized_shape))
            # Learnable shifting parameter β, initialized to 0
            self.beta = nn.Parameter(torch.zeros(*self.normalized_shape))
        else:
            # Without learnable parameters
            self.register_parameter('gamma', None)
            self.register_parameter('beta', None)
    
    def forward(self, x):
        # Compute the mean and variance
        mean = x.mean(dim=-1, keepdim=True)
        # Variance estimation:
        # Biased formula (denominator = n)
        # var = (1/n) * Σ(x_i - mean)²
        # Unbiased formula (denominator = n-1)
        # var = (1/(n-1)) * Σ(x_i - mean)²
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        
        # Normalization formula
        x_normalized = (x - mean) / torch.sqrt(variance + self.eps)
        
        # Apply learnable affine transformation
        if self.elementwise_affine:
            output = self.gamma * x_normalized + self.beta
        else:
            output = x_normalized
            
        return output


In [14]:
# Create input data: (batch_size, seq_len, features)
batch_size, seq_len, features = 16, 10, 512
x = torch.randn(batch_size, seq_len, features)

print("Input shape:", x.shape)
print("Output shape (before normalization):", x.shape)
print("Mean of input (per token):", x.mean(dim=-1))
print("Variance of input (per token):", x.var(dim=-1, unbiased=False))

Input shape: torch.Size([16, 10, 512])
Output shape (before normalization): torch.Size([16, 10, 512])
Mean of input (per token): tensor([[ 0.0549,  0.0744,  0.0314, -0.0353, -0.0475,  0.0580,  0.0316, -0.0085,
          0.0117,  0.0082],
        [ 0.0178, -0.0522,  0.0760, -0.0399, -0.0131,  0.0346, -0.0018, -0.0130,
          0.0291, -0.0201],
        [ 0.0988, -0.0112, -0.0114, -0.0418, -0.0629,  0.0842,  0.0898,  0.0089,
         -0.0912,  0.0127],
        [ 0.0399,  0.0516, -0.0548, -0.0281, -0.0237,  0.0342,  0.0397,  0.0064,
          0.0232, -0.0534],
        [-0.0411,  0.0049, -0.0133,  0.0279,  0.0027, -0.0268, -0.0327,  0.0045,
          0.0475,  0.0030],
        [-0.0505,  0.0722,  0.0067,  0.0325,  0.0098, -0.0194,  0.0025, -0.0062,
         -0.0202, -0.0112],
        [ 0.0897, -0.0397,  0.0545,  0.0177, -0.0373,  0.0286,  0.0111, -0.0630,
          0.0200, -0.0568],
        [ 0.0114, -0.0553, -0.0048,  0.0971,  0.0005, -0.0632, -0.0256,  0.0685,
         -0.0590, -0.0052],

In [15]:
# Instantiate the custom Layer Normalization module
custom_norm = CustomLayerNorm(features)

# Forward pass
output = custom_norm(x)

print("\nAfter Custom LayerNorm:")
print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Mean of output (per token):", output.mean(dim=-1))
print("Variance of output (per token):", output.var(dim=-1, unbiased=False))


After Custom LayerNorm:
Input shape: torch.Size([16, 10, 512])
Output shape: torch.Size([16, 10, 512])
Mean of output (per token): tensor([[ 7.4506e-09,  5.1223e-09, -1.5832e-08,  1.3039e-08,  4.1910e-09,
         -5.1223e-09,  7.9162e-09,  1.2573e-08,  1.1176e-08,  2.7940e-09],
        [-1.1176e-08, -3.7253e-09,  9.3132e-09,  1.8626e-09, -1.1176e-08,
         -1.5832e-08,  1.4901e-08, -1.6764e-08,  1.7229e-08,  2.0489e-08],
        [ 7.4506e-09, -2.2352e-08,  2.3283e-10, -1.2107e-08, -7.4506e-09,
          1.1176e-08, -1.1176e-08, -1.8626e-09,  7.4506e-09,  7.4506e-09],
        [-9.3132e-09, -3.2596e-09, -1.2107e-08, -1.2107e-08, -3.7253e-09,
         -1.8626e-09, -1.0245e-08, -4.6566e-09, -1.1176e-08,  9.3132e-10],
        [-7.4506e-09,  9.3132e-09, -1.0245e-08, -1.8626e-08, -1.9558e-08,
         -2.2352e-08, -2.3283e-09,  1.8626e-09,  1.3504e-08, -5.5879e-09],
        [-3.7253e-09, -1.3039e-08,  1.1176e-08,  2.3283e-09, -5.5879e-09,
          1.9325e-08,  1.6764e-08, -1.3970e-08,  

## Softmax implementation

In [ ]:
# Example of a multi-class network output layer
def softmax_torch(x, dim=-1):
    """Softmax function implemented in PyTorch.
    
    Args:
        x: Input tensor
        dim: The dimension along which to apply softmax
    
    Returns:
        A tensor after applying the softmax activation
    """
    # Numerical stability: subtract the max value to prevent overflow
    max_vals = torch.max(x, dim=dim, keepdim=True).values
    e_x = torch.exp(x - max_vals)
    
    return e_x / torch.sum(e_x, dim=dim, keepdim=True)

In [6]:
seq_len =  10
x = torch.randn(seq_len)

result = softmax_torch(x)
print(f"result is {result}")
print(f"sum of result is {sum(result)}")

result is tensor([0.0321, 0.0956, 0.0213, 0.0288, 0.1199, 0.0905, 0.1486, 0.2773, 0.1471,
        0.0388])
sum of result is 1.0


In [7]:
def softmax_temperature(x, temperature=1.0, dim=-1):
    """Softmax function with a temperature parameter.
    
    Effect of the temperature parameter:
        temperature > 1.0: smooths the distribution (increases entropy)
        temperature < 1.0: sharpens the distribution (reduces entropy)
    
    Args:
        x: Input tensor (logits)
        temperature: Scaling factor controlling distribution sharpness
        dim: The dimension along which to apply softmax
    
    Returns:
        A tensor representing the temperature-scaled softmax probabilities
    """
    e_x = torch.exp(x / temperature)
    return e_x / torch.sum(e_x, dim=dim, keepdim=True)

In [8]:
result = softmax_temperature(x, 2)
print(f"result is {result}")
print(f"sum of result is {sum(result)}")

result is tensor([0.0608, 0.1050, 0.0496, 0.0577, 0.1176, 0.1022, 0.1309, 0.1789, 0.1303,
        0.0670])
sum of result is 1.0000001192092896


In [9]:
result = softmax_temperature(x, 0.2)
print(f"result is {result}")
print(f"sum of result is {sum(result)}")

result is tensor([1.8688e-05, 4.3960e-03, 2.4183e-06, 1.0967e-05, 1.3621e-02, 3.3310e-03,
        3.9794e-02, 9.0089e-01, 3.7891e-02, 4.8626e-05])
sum of result is 1.0


In [10]:
def batched_softmax(x):
    """Softmax function for batched inputs.
    
    Input shape:  (batch_size, num_classes)
    Output shape: (batch_size, num_classes)
    
    Args:
        x: Input tensor containing logits for each sample in the batch.
    
    Returns:
        Tensor of the same shape as input, where each row sums to 1.
    """
    # Numerical stability: subtract the maximum value in each row
    max_vals, _ = torch.max(x, dim=1, keepdim=True)
    e_x = torch.exp(x - max_vals)
    
    return e_x / torch.sum(e_x, dim=1, keepdim=True)

In [17]:
batch_size, seq_len = 1, 10
x = torch.randn(batch_size, seq_len)
print(f"x is {x}")
result = batched_softmax(x)
print(f"result is {result}")
print(f"sum of result is {torch.sum(result, -1)}")

x is tensor([[ 0.4345,  0.7254,  1.5437,  1.3455,  0.5254,  0.0400, -0.3758,  1.6476,
          1.4629, -0.2691]])
result is tensor([[0.0598, 0.0800, 0.1813, 0.1487, 0.0655, 0.0403, 0.0266, 0.2011, 0.1672,
         0.0296]])
sum of result is tensor([1.0000])
